In [ ]:
!pip install ultralytics

In [ ]:
!pip install roboflow

import os
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("bere-gcgue").project("smartphones_capstone")
dataset = project.version(4).download("yolov11")

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")  # small model, good balance for Colab T4

results = model.train(
    data="/content/smartphones_capstone-4/data.yaml",
    epochs=120,
    imgsz=640,
    batch=16,
    cos_lr=True,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    patience=30
)

In [ ]:
!ls /content

In [ ]:
# Evaluate model performance on the validation set
metrics = model.val()

precision = metrics.box.mp
recall = metrics.box.mr

f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

print("\n=== FINAL TRAINING METRICS ===")
print(f"mAP50:     {metrics.box.map50 * 100:.2f}%")
print(f"mAP50-95:  {metrics.box.map * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-Score:  {f1 * 100:.2f}%")

In [ ]:
from ultralytics import YOLO
model = YOLO("/content/runs/detect/train-3/weights/best.pt"
)
model.export(format="onnx", imgsz=640, simplify=True)